# C02. What the lock was actually protecting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/c02-what-the-lock-was-protecting/c02.ipynb)

"The GIL makes Python thread safe" is one sentence covering two completely different claims, and only one of them was ever true.

This lesson separates them, using a counter that four threads add to. On the build you are almost certainly running, that counter comes out exactly right. Change one thing that looks like it could not possibly matter and it stops coming out right, with the lock still on.

![the interpreter's own data structures on one side and your own variables on the other, with only the first one actually promised](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/two-kinds-of-safety.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/cpython/pylock.h:12-35@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Most of this lesson runs anywhere. Two things do not: a browser tab cannot start a thread, and it cannot start another process either. Those cells check first and say so rather than printing nonsense. In Colab or from a checkout, everything runs.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## A counter that comes out right

Here is the oldest threading exercise there is. Four threads, a shared number, each thread adds one to it a hundred thousand times. The right answer is four hundred thousand.

In most languages you would get some number smaller than that, because `counter = counter + 1` is three steps: read it, add one, write it back. If another thread reads between your read and your write, one of the two increments disappears. That is a [race condition](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#race-condition), and it is the first thing anybody learns about threads.

Run it on this Python and you get exactly four hundred thousand. Every time.

The cell turns the [switch interval](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#switch-interval) down to a microsecond first. C01 spent a section on that number: it is how long a thread waiting for the [GIL](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#gil) sits patiently before it asks for it. Turning it down means threads hand the lock over as often as they possibly can, which is the setting most likely to lose an increment. It still does not.

Four threads each adding one to the same global a hundred thousand times end up with exactly four hundred thousand on an ordinary build, even with the switch interval turned all the way down

In [ ]:
import threading

ROUNDS = 100_000
THREADS = 4
WANT = ROUNDS * THREADS
NO_THREADS = "  this build cannot start a thread, so there is nothing to race here"


def threads_work():
    """Some builds cannot start a thread at all. A browser tab is one of them."""
    try:
        probe = threading.Thread(target=lambda: None)
        probe.start()
        probe.join()
    except RuntimeError:
        return False
    return True


THREADS_WORK = threads_work()
counter = 0


def race(target):
    """Run `target` on THREADS threads with the switch interval as low as it will go."""
    global counter
    before = sys.getswitchinterval()
    sys.setswitchinterval(0.000001)
    try:
        counter = 0
        hands = [threading.Thread(target=target) for _ in range(THREADS)]
        for hand in hands:
            hand.start()
        for hand in hands:
            hand.join()
    finally:
        sys.setswitchinterval(before)
    return counter


def plain():
    global counter
    for _ in range(ROUNDS):
        counter = counter + 1


if not THREADS_WORK:
    print(NO_THREADS)
else:
    print(f"  asked for {WANT:,} increments")
    print(f"  ended up with {race(plain):,}")

> **Version note.** on an ordinary build this comes out exactly right every time, and on a free threaded build it comes out short, because nothing here is protecting the counter and the two builds differ in whether anything else is

## The same counter, one call further apart

Now change one thing. Instead of `counter = counter + 1`, write `counter = add_one(counter)`, where `add_one` returns its argument plus one. Same arithmetic, same threads, same interval, same lock.

It stops coming out right. Roughly half the increments go missing.

Then a third version, which reads the counter, runs a loop that goes round exactly once and does nothing, and writes the value back. That one loses even more.

![three ways of writing the same increment, what sits between the read and the write in each, and how many increments survive](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/where-the-count-goes-wrong.svg)

The reason is the whole of C01 in one line. A running thread only gives up the GIL at a [periodic check](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#periodic-check), and the compiler emits one of those at backward jumps and at function resumes and nowhere else [Python/bytecodes.c:158-161@v3.15.0rc1#_CHECK_PERIODIC](https://github.com/python/cpython/blob/v3.15.0rc1/Python/bytecodes.c#L158-L161). `counter = counter + 1` has neither, so no other thread can get in partway through it. Add a call and there is a function resume. Add a loop and there is a backward jump. Now there is somewhere for the lock to move, and increments start disappearing.

So the counter was never safe. It was unreachable, which looks the same right up until you refactor the line.

Putting a function call or a one pass loop between the read and the write makes the same counter start losing increments, on the same build with the same lock

In [ ]:
def add_one(value):
    return value + 1


def through_a_call():
    global counter
    for _ in range(ROUNDS):
        counter = add_one(counter)


def with_a_loop():
    global counter
    for _ in range(ROUNDS):
        value = counter
        for _ in range(1):
            pass
        counter = value + 1


shapes = (
    ("nothing in between", plain),
    ("a call in between", through_a_call),
    ("a loop in between", with_a_loop),
)

if not THREADS_WORK:
    print(NO_THREADS)
else:
    for name, work in shapes:
        got = race(work)
        print(f"  {name:<20} {got:>8,} of {WANT:,}   {got / WANT:>6.0%} survived")

> **Version note.** the first row is exact on any build with a GIL and short on a free threaded one, and the other two are short everywhere, but how short depends on how the operating system schedules the four threads on this particular machine

## The list that never loses anything

Try the same shape with a list. Four threads, four hundred thousand `append` calls each, all into one list. Count what ended up in it.

You get all of them. Not most of them, all of them, and not by luck. `list.append` grows an array and writes into it, which is a read and a write of the list's internals, and two threads doing that at once could easily leave one item on the floor or the length wrong.

This one is a promise, and it is a promise on both builds. On the build with the [GIL](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#gil) it holds because only one thread runs at a time. On the [free threaded build](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#free-threaded-build) it holds because the append is wrapped in a [critical section](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#critical-section) that takes that particular list's own lock [Objects/listobject.c:538-550@v3.15.0rc1#PyList_Append](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/listobject.c#L538-L550).

The nicest part is where that wrapping comes from. The method is declared in an [Argument Clinic](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#argument-clinic) block with the line `@critical_section` above it [Objects/listobject.c:1222-1239@v3.15.0rc1#list_append_impl](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/listobject.c#L1222-L1239), and a script turns that one word into the C that takes the lock [Objects/clinic/listobject.c.h:115-126@v3.15.0rc1#Py_BEGIN_CRITICAL_SECTION](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/clinic/listobject.c.h#L115-L126). B04 is the lesson about how much of CPython is written that way, and this is one of the better examples of why.

Four threads appending to the same list lose nothing, on any build, because the append itself takes a lock

In [ ]:
APPENDS = 400_000


def fill(target):
    for _ in range(APPENDS):
        target.append(1)


if not THREADS_WORK:
    print(NO_THREADS)
else:
    shared = []
    fillers = [threading.Thread(target=fill, args=(shared,)) for _ in range(THREADS)]
    for filler in fillers:
        filler.start()
    for filler in fillers:
        filler.join()
    print(f"  appends asked for {APPENDS * THREADS:>10,}")
    print(f"  items in the list {len(shared):>10,}")

> **Version note.** the two numbers are equal on every build, and the only thing that changes between builds is how long the cell takes to run

## The lock that fits in one byte

So where does a list keep its lock. In the object itself.

O01 took the [object header](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#object-header) apart and found the free threaded one is twice the size, with an `ob_mutex` field in it [Include/object.h:156-167@v3.15.0rc1#ob_mutex](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L156-L167). That field is a `PyMutex`, and a `PyMutex` is one byte [Include/cpython/pylock.h:12-35@v3.15.0rc1#PyMutex](https://github.com/python/cpython/blob/v3.15.0rc1/Include/cpython/pylock.h#L12-L35).

Two bits of that byte are used. One says whether somebody is holding it. The other says whether anybody is parked waiting for it.

![the four states of the one byte mutex and what a thread does in each](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/one-byte-of-lock.svg)

Taking a free one is a single compare and exchange instruction, with no system call and no allocation [Include/cpython/pylock.h:46-59@v3.15.0rc1#_PyMutex_Lock](https://github.com/python/cpython/blob/v3.15.0rc1/Include/cpython/pylock.h#L46-L59). That is the whole reason this design is possible. A lock cheap enough to take and release millions of times a second is a lock you can afford to put inside every object, instead of one big one you take at the door.

Every object on the free threaded build carries its own one byte lock, and taking an uncontended one is a single instruction

That is a [per object lock](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#per-object-lock). The rest of the design is about what you do when you need two of them.

## Two objects at once

Locks that live in objects have the usual problem. Thread A takes list `a` and then wants `b`. Thread B takes `b` and then wants `a`. Nobody moves again.

The usual fix is a rule about the order locks are taken in, which somebody eventually breaks. CPython does something else. A [critical section](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#critical-section) is allowed to let go of its lock partway through, and an inner one suspends the outer ones instead of nesting inside them [Include/critical_section.h:7-22@v3.15.0rc1#Py_BEGIN_CRITICAL_SECTION2](https://github.com/python/cpython/blob/v3.15.0rc1/Include/critical_section.h#L7-L22). A thread therefore never sits waiting for one lock while holding another, so there is no cycle to get stuck in.

![two threads deadlocking on nested locks, against the pair of locks taken together](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/locking-two-at-once.svg)

Which does mean you cannot nest two of them and expect to hold both. So there is a separate macro for that case, and `list_a + list_b` is one of the places that uses it [Objects/listobject.c:810-816@v3.15.0rc1#Py_BEGIN_CRITICAL_SECTION2](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/listobject.c#L810-L816).

And on your build all of this compiles to nothing at all. `Py_BEGIN_CRITICAL_SECTION` is redefined as an opening brace and `Py_END_CRITICAL_SECTION` as a closing one [Include/cpython/critical_section.h:44-61@v3.15.0rc1#Py_BEGIN_CRITICAL_SECTION](https://github.com/python/cpython/blob/v3.15.0rc1/Include/cpython/critical_section.h#L44-L61). The GIL is already doing the job, so the annotations are free. That is what lets one source tree serve both builds.

## One lock each, or one lock between them

Per object locks have a consequence you can measure. If four threads append to four different lists, they take four different locks and never wait for each other. If four threads append to the same list, they all queue on one byte.

On the build you have, those two cases are the same measurement, because the GIL makes them the same. The cell below shows that: both come out around the same time and neither is faster than one thread doing all the work.

On a build with the GIL, four threads sharing one list and four threads with a list each take about the same time as each other

In [ ]:
import time

TRIES = 5


def one_list(count):
    """One list, handed to every thread, so every append lands on the same object."""
    shared = []
    return [shared] * count


def a_list_each(count):
    """A list per thread, so no two threads ever want the same lock."""
    return [[] for _ in range(count)]


def best(make_targets, count):
    """Fastest wall clock time out of TRIES runs, because one timing is a coin toss."""
    times = []
    for _ in range(TRIES):
        targets = make_targets(count)
        crew = [threading.Thread(target=fill, args=(target,)) for target in targets]
        start = time.perf_counter()
        for hand in crew:
            hand.start()
        for hand in crew:
            hand.join()
        times.append(time.perf_counter() - start)
    return min(times)


if not THREADS_WORK:
    print(NO_THREADS)
else:
    fill([])
    one = best(a_list_each, 1)
    print(f"  one thread, one list        {one * 1000:>7.0f} ms")
    for label, make in (("the same list", one_list), ("a list each", a_list_each)):
        took = best(make, THREADS)
        rate = THREADS * one / took
        print(f"  {THREADS} threads, {label:<14} {took * 1000:>7.0f} ms   {rate:.2f}x")

> **Version note.** the times depend on the machine, and on an ordinary build the two speedup figures land near each other and well under one, because only one thread is appending at a time whichever list it is appending to

Now the same program on the free threaded image, where the locks are real.

![one thread against four sharing a list against four with a list each, as a bar chart of milliseconds](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/same-list-or-a-list-each.svg)

Four threads with their own lists get most of the parallel speedup. Four threads sharing one list are slower than one thread doing all the work, because every append now costs a contended lock on top of the work.

Without the GIL, four threads appending to four lists get real speedup while four threads appending to one list are slower than one thread doing all of it

Does a lock in every object mean four threads with four lists go faster than four threads with one?

```python
"""Four threads appending to one list, and four threads with a list each.

The lesson runs this on a build with a GIL, where the two cases come out as the same
measurement, because only one thread is running Python either way. What that build cannot show
is the locking underneath, since the critical section around list.append compiles to a pair of
braces there and costs nothing.

On a build configured with --disable-gil the two cases stop being the same. Every list carries
its own one byte mutex in its object header, so four threads appending to four lists take four
different locks and never wait for each other, while four threads appending to one list all
queue on one byte.

Everything here is the best of nine runs after a warmup, because this image runs under emulation
on a virtual machine with a handful of shared cores, and a run that happens to land while the
host is busy comes out several times slower than the same run on a quiet machine.
"""

import sys
import threading
import time

ROUNDS = 400_000
THREADS = 4
TRIES = 9


def fill(target):
    for _ in range(ROUNDS):
        target.append(1)


def one_list(count):
    """One list, handed to every thread, so every append lands on the same object."""
    shared = []
    return [shared] * count


def a_list_each(count):
    """A list per thread, so no two threads ever want the same lock."""
    return [[] for _ in range(count)]


def best(make_targets, count):
    """Fastest wall clock time out of TRIES runs of the work on `count` threads."""
    times = []
    for _ in range(TRIES):
        targets = make_targets(count)
        threads = [threading.Thread(target=fill, args=(target,)) for target in targets]
        start = time.perf_counter()
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
        times.append(time.perf_counter() - start)
    return min(times)


print(f"sys._is_gil_enabled() reports: {sys._is_gil_enabled()}")
print()

fill([])
one = best(a_list_each, 1)
print(f"~ one thread appending to one list: {one * 1000:.0f} ms")

for label, make in (("the same list", one_list), ("a list each", a_list_each)):
    took = best(make, THREADS)
    print(f"~ {THREADS} threads appending to {label}: {took * 1000:.0f} ms")
    print(f"~ speedup over one thread doing all of it: {THREADS * one / took:.2f}x")

total = []
racers = [threading.Thread(target=fill, args=(total,)) for _ in range(THREADS)]
for racer in racers:
    racer.start()
for racer in racers:
    racer.join()

print()
print(f"appends asked for: {ROUNDS * THREADS}")
print(f"items in the list: {len(total)}")
```

```text
sys._is_gil_enabled() reports: False

~ one thread appending to one list: 6 ms
~ 4 threads appending to the same list: 45 ms
~ speedup over one thread doing all of it: 0.52x
~ 4 threads appending to a list each: 10 ms
~ speedup over one thread doing all of it: 2.44x

appends asked for: 1600000
items in the list: 1600000
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

## The lock can come back

One more thing, and it is the part that surprises people who assume free threading is a one way door.

A free threaded interpreter can turn its GIL back on. There are two ways in. The first is a flag at startup, `-X gil=1` or the `PYTHON_GIL` environment variable [Python/initconfig.c:1970-1990@v3.15.0rc1#PYTHON_GIL](https://github.com/python/cpython/blob/v3.15.0rc1/Python/initconfig.c#L1970-L1990). The same parser reads both, and it is worth knowing what it does on your build: `-X gil=1` is quietly accepted and does nothing, and `-X gil=0` is a hard error, because your interpreter has no GIL to remove.

On an ordinary build, -X gil=1 is accepted and does nothing while -X gil=0 refuses to start the interpreter at all

In [ ]:
import subprocess

ASK = "import sys; print('started, and the lock is on:', sys._is_gil_enabled())"


def start_python(setting):
    """Start another copy of this interpreter with -X gil set, and report what it said."""
    if not sys.executable:
        return None
    try:
        done = subprocess.run(
            [sys.executable, "-X", f"gil={setting}", "-c", ASK],
            capture_output=True,
            text=True,
            timeout=60,
        )
    except OSError:
        return None
    return (done.stdout + done.stderr).strip().splitlines()[0]


for setting in ("1", "0"):
    said = start_python(setting)
    if said is None:
        print("  this build cannot start another process, so there is nothing to try here")
        break
    print(f"  -X gil={setting}  {said}")

> **Version note.** on an ordinary build the first line starts the interpreter and the second is a fatal error, and on a free threaded build both start and report the lock as on and off respectively

The second way in is stranger, and it happens after the interpreter is already running.

An extension module written in C declares whether it is safe without the lock, using a slot called `Py_mod_gil` [Include/moduleobject.h:85-89@v3.15.0rc1#Py_MOD_GIL_NOT_USED](https://github.com/python/cpython/blob/v3.15.0rc1/Include/moduleobject.h#L85-L89). Module setup reads that slot and remembers the answer [Objects/moduleobject.c:471-476@v3.15.0rc1#Py_mod_gil](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/moduleobject.c#L471-L476). If a module does not say, the import machinery assumes the worst and turns the GIL on for the whole interpreter, warning as it goes [Python/import.c:1618-1643@v3.15.0rc1#_PyImport_CheckGILForModule](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L1618-L1643).

![an extension import with no Py_mod_gil slot leading to the GIL being switched on for the rest of the run](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c02-what-the-lock-was-protecting/diagrams/the-lock-coming-back.svg)

That switch is permanent for the rest of the process [Python/import.c:1645-1665@v3.15.0rc1#_PyImport_EnableGILAndWarn](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L1645-L1665). The counter it keeps goes to `INT_MAX` and never comes down [Python/ceval_gil.c:1132-1150@v3.15.0rc1#_PyEval_EnableGILPermanent](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_gil.c#L1132-L1150), which is the difference between this and the ordinary case where the count goes up and down as modules that need the lock are loaded and dropped [Python/ceval_gil.c:1152-1191@v3.15.0rc1#_PyEval_DisableGIL](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_gil.c#L1152-L1191).

So one unported extension deep in your dependencies can put the lock back, and the only sign is a `RuntimeWarning` on import. `sys._is_gil_enabled()` is worth calling after your imports rather than before them.

The recording runs the three counters from earlier on the free threaded image, twice, in child processes of the same binary. Once with the lock off, where all three lose increments. Once with `-X gil=1`, where the first one goes back to being exactly right and the other two do not.

The same free threaded binary started with -X gil=1 gets the exact answer for the plain counter back, and still loses increments for the other two shapes

What happens to a racing counter when the same free threaded binary is started with the lock back on?

```python
"""The same racing counter on one binary, with the lock off and then back on.

A free threaded build does not have to stay free threaded. Starting it with -X gil=1 turns the
lock on before anything runs, and an extension module that has not declared itself safe turns
the lock on during its own import, while the interpreter is already going.

So this program runs the same three counters twice, in child processes of itself, once with the
lock off and once with it on. The three differ only in what sits between reading the variable
and writing it back: nothing at all, a function call, or a loop that goes round once.

The lesson's point is that the exact answer on an ordinary build, four hundred thousand out of
four hundred thousand, is a property of where the interpreter is allowed to hand the lock over
rather than of the assignment being one step. With the lock off, none of the three is safe.
With the same binary and the lock back on, the first one is and the other two are not.

The child turns the switch interval right down, the same as the lesson's cell does, so that the
handoffs happen often enough to see in a run this short.
"""

import subprocess
import sys

CHILD = """
import sys
import threading

sys.setswitchinterval(0.000001)

ROUNDS = 100_000
THREADS = 4
counter = 0


def add_one(value):
    return value + 1


def plain():
    global counter
    for _ in range(ROUNDS):
        counter = counter + 1


def through_a_call():
    global counter
    for _ in range(ROUNDS):
        counter = add_one(counter)


def with_a_loop():
    global counter
    for _ in range(ROUNDS):
        value = counter
        for _ in range(1):
            pass
        counter = value + 1


def go(target):
    global counter
    counter = 0
    hands = [threading.Thread(target=target) for _ in range(THREADS)]
    for hand in hands:
        hand.start()
    for hand in hands:
        hand.join()
    return counter


shapes = (
    ("nothing in between", plain),
    ("a call in between", through_a_call),
    ("a loop in between", with_a_loop),
)

print(f"  the lock is on: {sys._is_gil_enabled()}")
for name, work in shapes:
    print(f"~   {name}: {go(work)} of {ROUNDS * THREADS}")
"""

print(f"this process itself started with the lock off: {not sys._is_gil_enabled()}")

for setting in ("0", "1"):
    print()
    print(f"the same binary, started with -X gil={setting}")
    done = subprocess.run(
        [sys.executable, "-X", f"gil={setting}", "-c", CHILD],
        capture_output=True,
        text=True,
        check=True,
    )
    print(done.stdout, end="")
```

```text
this process itself started with the lock off: True

the same binary, started with -X gil=0
  the lock is on: False
~   nothing in between: 160723 of 400000
~   a call in between: 135319 of 400000
~   a loop in between: 139300 of 400000

the same binary, started with -X gil=1
  the lock is on: True
~   nothing in between: 400000 of 400000
~   a call in between: 176504 of 400000
~   a loop in between: 135230 of 400000
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

## Try it yourself

Four things, roughly in order of how much they will teach you.

Take the `with_a_loop` version and delete the inner loop, leaving the read and the write on separate lines with nothing between them. It goes back to being exactly right. Two statements are no less atomic than one, because atomicity was never the thing doing the work.

Put `counter += 1` in as a fourth shape. It is the same as `plain` on an ordinary build, and it is worth confirming that yourself rather than taking it on trust, because the `+=` spelling is the one people assume is special.

Swap the list in the append cell for a `dict`, setting `d[i] = i` from four threads with non overlapping ranges of `i`. Count the keys at the end. Dicts carry the same annotations as lists, so the count is exact, and O07 is the lesson about what is being protected in there.

Wrap the increment in a `threading.Lock`. Every shape becomes exact on every build, and the cell gets a lot slower. That is the actual answer, and it always was: if two threads share a variable, lock it yourself.

## What you now know

"The GIL makes Python thread safe" is two claims. The interpreter's own data structures are safe, and always were, and still are. Your own variables never were.

A counter incremented by four threads comes out exactly right on an ordinary build, and stops being right the moment a function call or a loop appears between the read and the write. Nothing about the lock changed. The only thing that changed is whether there is a periodic check in the middle, which is where the interpreter is allowed to hand the lock over.

Free threading replaced the one big lock with a one byte mutex in every object header, cheap enough to take millions of times a second because an uncontended one is a single instruction.

Deadlock is avoided by letting a critical section suspend itself rather than by ordering the locks, and by a separate macro for the cases that genuinely need two objects at once. On a build with the GIL all of it compiles to a pair of braces.

Locks in objects means contention is per object. Four threads on four lists scale. Four threads on one list are slower than one thread.

And the lock can come back, either from a flag at startup or from importing one C extension that has not declared itself safe, which turns it on permanently for the whole interpreter with nothing but a warning.

## What is next

C03 goes down one more level, to what a thread actually is to the interpreter. Every one of them has a `PyThreadState`, they hang off the interpreter in a list, and attaching and detaching from that list is what the GIL handoff in C01 was really doing.